# multiply-back composite — cx2: multiply_back0/back1 — per-arg chain rule + unbroadcast + argnum dispatch

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `multiply-back`, `unbroadcast-pattern`, `arg-position-back-functions`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "multiply-back"
DD_ATOM_IDS = ["multiply-back", "unbroadcast-pattern", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: multiply_back", "Backprop: Unbroadcast pattern", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Binary-op back fns — three atoms in one expression

Every binary op `out = f(x, y)` needs TWO back fns — one per arg-position —
and each one ends with an unbroadcast step. Composition:

1. **`arg-position-back-functions`** — argnum=0 returns `dL/dx`, argnum=1
   returns `dL/dy`. For `multiply`, the two bodies differ only in which parent
   appears: `grad_out * y` vs `grad_out * x`.
2. **`multiply-back`** — the per-arg chain rule: `d(x*y)/dx = y`, so the raw
   chain step is `grad_out * y` (and symmetrically for arg-1).
3. **`unbroadcast-pattern`** — wrap the chain-rule result in `unbroadcast(grad,
   parent)` so it matches the *pre-broadcast* parent shape. Without this, the
   grad would have the broadcasted (output) shape and node `x` couldn't
   accumulate it.

The whole composite collapses to one expression per arg:
`unbroadcast(grad_out * other_parent, this_parent)`.


### Composite Exercise — multiply_back0/back1 — per-arg chain rule + unbroadcast + argnum dispatch

**Atoms exercised together**: `multiply-back`, `unbroadcast-pattern`, `arg-position-back-functions`

Build a single `cx2_multiply_back(grad_out, out, x, y, argnum)` that dispatches
on `argnum` (0 or 1) and returns the gradient w.r.t. the requested parent,
**already unbroadcast** to the parent's original shape.

Use the provided `unbroadcast(grad, original)` helper (in the stub).

Required behaviour:
- `argnum=0` → return `dL/dx`, shape == `x.shape`.
- `argnum=1` → return `dL/dy`, shape == `y.shape`.
- Any other `argnum` → `raise ValueError`.
- Broadcasting case: when `x.shape != y.shape != out.shape`, the returned grad
  must be summed along the axes that were broadcast.


In [ ]:
def unbroadcast(grad, original):
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def cx2_multiply_back(grad_out, out, x, y, argnum):
    if argnum == 0:
        return unbroadcast(grad_out * y, x)   # dL/dx = grad_out * y, then unbroadcast
    if argnum == 1:
        return unbroadcast(grad_out * x, y)   # dL/dy = grad_out * x, then unbroadcast
    raise ValueError(f'argnum must be 0 or 1, got {argnum}')


<details><summary>Show solution — cx2</summary>

```python
def unbroadcast(grad, original):
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def cx2_multiply_back(grad_out, out, x, y, argnum):
    if argnum == 0:
        return unbroadcast(grad_out * y, x)   # dL/dx = grad_out * y, then unbroadcast
    if argnum == 1:
        return unbroadcast(grad_out * x, y)   # dL/dy = grad_out * x, then unbroadcast
    raise ValueError(f'argnum must be 0 or 1, got {argnum}')

```

All three atoms live in the same expression: the `argnum` branch is the
arg-position dispatch; `grad_out * y` (or `* x`) is the multiply-back chain
rule; `unbroadcast(..., parent)` is the unbroadcast pattern. Dropping any one
of them breaks at least one of the (b), (c)/(d), (e) test cases.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx2',
        'subtopics': ["Backprop: multiply_back", "Backprop: Unbroadcast pattern", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()